In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import os
import tensorflow as tf

####################################
#Multi_gpu usage strategy run code.#
####################################

# gpus = tf.config.experimental.list_physical_devices('GPU')
# if gpus:
#     try:
#         print(gpus[0])
#         print(gpus[1])
#         print(len(gpus), "Physical GPUs,", len(gpus), "Logical GPUs")
#     except RuntimeError as e:
#         print(e)  
# else:
#     print("No GPUs found")

# strategy = tf.distribute.MirroredStrategy(devices=["/GPU:0", "/GPU:1"])
# print(f'Number of replicas: {strategy.num_replicas_in_sync}')


strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")

In [ ]:
from tensorflow.keras import layers, Model, optimizers
    


# Define a basic residual block for both downsampling and upsampling
def residual_block(x, filters, downsample=False, upsample=False):
    identity = x

    if downsample:
        # Downsample identity to match the output shape
        identity = layers.Conv2D(filters, (1, 1), strides=(2, 2), padding='same')(identity)
        x = layers.Conv2D(filters, (5, 5), strides=(2, 2), padding='same')(x)
    elif upsample:
        # Upsample identity to match the output shape
        identity = layers.Conv2DTranspose(filters, (1, 1), strides=(2, 2), padding='same')(identity)
        x = layers.Conv2DTranspose(filters, (7, 7), strides=(2, 2), padding='same')(x)
    else:
        # Adjust the identity tensor if filters don't match
        if identity.shape[-1] != filters:
            identity = layers.Conv2D(filters, (1, 1), padding='same')(identity)
        
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    if not upsample:
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    else:
        x = layers.Conv2DTranspose(filters, (5, 5), padding='same')(x)
        
    x = layers.BatchNormalization()(x)
    
    # Residual connection
    x = layers.Add()([x, identity])
    x = layers.ReLU()(x)
    return x

# Define the Backbone Net like an Autoencoder with Residual Blocks
def BackboneNet(input_tensor):
    # Encoding part (downsampling)
    x = layers.Conv2D(64, (7, 7), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Downsampling with residual blocks
    x = residual_block(x, 32, downsample=True)
    x = residual_block(x, 64, downsample=True)
    
    # Bottleneck layer
    x = residual_block(x, 128)
    
    # Decoding part (upsampling)
    x = residual_block(x, 64, upsample=True)
    x = residual_block(x, 32, upsample=True)
    
    # Output layer to get (200, 200, 1)
    x = layers.Conv2DTranspose(1, (3, 3), strides=(2, 2), padding='same', activation='sigmoid')(x)
    
    return x

# CBR Block Definition
def CBR_Block(x, filters, kernel_size=3, strides=1, padding='same'):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

# Input shapes
W, H = 128,128
with strategy.scope():
    # DEM Input (W, H, 1)
    dem_input = tf.keras.Input(shape=(W, H, 1), name='DEM_Input')
    dem_feature = CBR_Block(dem_input, filters=64)
    dem_feature = layers.Conv2D(64, (4, 4), strides=2, padding='same')(dem_feature)  # Downsample to 100x100x64

    # Fuel Input (W, H, 1)
    fuel_input = tf.keras.Input(shape=(W, H, 1), name='Fuel_Input')
    fuel_feature = CBR_Block(fuel_input, filters=64)
    fuel_feature = layers.Conv2D(64, (4, 4), strides=2, padding='same')(fuel_feature)  # Downsample to 100x100x64

    # Wind Field Input (2, 1) which needs to be reshaped to (W, H, 1)
    wind_vector_input = tf.keras.Input(shape=(2, 1), name='Wind_Vector_Input')
    # Reshape the (2, 1) wind vector to (W, H, 1)
    wind_feature = layers.Dense(W * H, activation='relu')(tf.keras.layers.Flatten()(wind_vector_input))
    wind_feature = layers.Reshape((W, H, 1))(wind_feature)
    wind_feature = CBR_Block(wind_feature, filters=64)
    wind_feature = layers.Conv2D(64, (4, 4), strides=2, padding='same')(wind_feature)  # Downsample to 100x100x64

    # T&H Input (1, 2)
    th_input = tf.keras.Input(shape=(1, 2), name='TH_Input')
    th_feature = layers.Dense(16, activation='relu')(th_input)
    th_feature = layers.Dense(192, activation='relu')(th_feature)  # (1, 1, 192)
    th_feature = layers.Lambda(lambda x: tf.expand_dims(x, axis=1))(th_feature)   # Reshape to (1, 1, 192)

    # Concatenate DEM, Fuel, Wind Field Features
    combined_spatial_features = layers.Concatenate()([dem_feature, fuel_feature, wind_feature])

    # Fusion: Element-wise multiplication of concatenated features and th_feature
    fusion_feature = layers.Multiply()([combined_spatial_features, th_feature])

    # Add residual connection: Combine fusion result with the original concatenated spatial features
    residual_combined = layers.Add()([combined_spatial_features, fusion_feature])

    # CBR Layer after fusion to downsample to 100x100x64
    condition_feature = CBR_Block(residual_combined, filters=64)

    # Pass through the ResNet18 backbone (Functional API version)
    resnet_output = BackboneNet(condition_feature)


    # Final Conv Layer for SSTDF Prediction
    sstdf_prediction = layers.Conv2D(1, (1, 1), activation='tanh', name='SSTDF_Prediction')(resnet_output)

    # SSTDF Label Input (for loss function)
    sstdf_label = tf.keras.Input(shape=(W, H, 1), name='SSTDF_Label')

    # Model Definition
    model = Model(inputs=[ dem_input, fuel_input, wind_vector_input, th_input], outputs=resnet_output)


    # Compile the model
    model.compile(optimizer='adam', loss=tf.keras.losses.MeanAbsoluteError(), metrics=['accuracy'])

# Print the model summary
model.summary()





In [ ]:
import pydot
print(pydot.find_graphviz())

from tensorflow.keras.utils import plot_model
plot_model(model, to_file='./resnet_film_generator.png',
           show_shapes=True,
           expand_nested=True)
from IPython.display import Image
Image('./resnet_film_generator.png')

In [ ]:
def _parse_function(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [128, 128, 1])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [128, 128, 3])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [128, 128, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [128, 128, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    extracted_tnh_element = tnh[0] # 첫 번째 시간의 요소 (2,)
    tnh_final = tf.expand_dims(extracted_tnh_element, axis=0) # (1, 2)
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    extracted_wind_element = wind[0] # 첫 번째 시간의 요소 (2,)
    wind_final = tf.expand_dims(extracted_wind_element, axis=-1) # (2, 1)
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return ((spread_12h+1)/2, (dem+1)/2, (fuel+1)/2,wind_final, tnh_final , ignite_index, weather_index)

def _parse_function2(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [128, 128, 1])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [128, 128, 3])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [128, 128, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [128, 128, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    extracted_tnh_element = tnh[0] # 첫 번째 시간의 요소 (2,)
    tnh_final = tf.expand_dims(extracted_tnh_element, axis=0) # (1, 2)
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    extracted_wind_element = wind[0] # 첫 번째 시간의 요소 (2,)
    wind_final = tf.expand_dims(extracted_wind_element, axis=-1) # (2, 1)
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return (spread_12h, dem, fuel,wind_final, tnh_final , ignite_index, weather_index)

def _parse_gan_function(
    record,
    spread_shape=(256,256,1),
    terrain_shape=(256,256,3),
    weather_img_shape=(256,256,1),
    weather_ws_shape=(2,1),
    weather_temp_hume_shape=( 1,2)
):
    feature_desc = {
        'fire_state':       tf.io.FixedLenFeature([], tf.string),
        'dem_image':        tf.io.FixedLenFeature([], tf.string),
        'fuel_image':       tf.io.FixedLenFeature([], tf.string),
        'terrain_image':    tf.io.FixedLenFeature([], tf.string),
        'ws_wd':            tf.io.FixedLenFeature([], tf.string),
        'temp_hume':        tf.io.FixedLenFeature([], tf.string),
        'weather_temp_img': tf.io.FixedLenFeature([], tf.string),
        'weather_hume_img': tf.io.FixedLenFeature([], tf.string),
        'weather_ws_img':   tf.io.FixedLenFeature([], tf.string),
        'weather_wd_img':   tf.io.FixedLenFeature([], tf.string),
        'spread_image':     tf.io.FixedLenFeature([], tf.string),
        'ignite_index':     tf.io.FixedLenFeature([], tf.int64),
        'weather_index':    tf.io.FixedLenFeature([], tf.int64),
    }
    parsed = tf.io.parse_single_example(record, feature_desc)

    def _decode_and_reshape(name, dtype, shape):
        raw = parsed[name]
        decoded = tf.io.decode_raw(raw, dtype)
        return tf.reshape(decoded, shape)

    fire_state       = _decode_and_reshape('fire_state',       tf.float32, spread_shape)
    spread_image     = _decode_and_reshape('spread_image',     tf.float32, spread_shape)
    dem_image        = _decode_and_reshape('dem_image',        tf.float32, spread_shape)
    fuel_image       = _decode_and_reshape('fuel_image',       tf.float32, spread_shape)
    terrain_image    = _decode_and_reshape('terrain_image',    tf.float32, terrain_shape)

    ws_wd            = _decode_and_reshape('ws_wd',            tf.float32, weather_ws_shape)
    temp_hume        = _decode_and_reshape('temp_hume',        tf.float32, weather_temp_hume_shape)

    weather_temp_img = _decode_and_reshape('weather_temp_img', tf.float32, weather_img_shape)
    weather_hume_img = _decode_and_reshape('weather_hume_img', tf.float32, weather_img_shape)
    weather_ws_img   = _decode_and_reshape('weather_ws_img',   tf.float32, weather_img_shape)
    weather_wd_img   = _decode_and_reshape('weather_wd_img',   tf.float32, weather_img_shape)

    # --- Indices ---
    ignite_index     = parsed['ignite_index']
    weather_index    = parsed['weather_index']

    return (
        spread_image,
        dem_image,
        fuel_image,
        ws_wd,
        temp_hume,
        ignite_index,
        weather_index
    )



In [ ]:

tfrecord_files = ["./train_mid_128_all.tfrecord"]
#tfrecord_files = ["./ignition/test_128_56ea.tfrecord"]

# # 데이터셋 생성 및 매핑
# train_dataset = tf.data.TFRecordDataset(tfrecord_files)
# train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
# #train_dataset = train_dataset.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
# train_dataset = train_dataset.batch(16, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
# train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)


def split_features_label(fire_state, dem,fuel, wind,tnh,ig,w ):
    # features는 fire_state, dem, fuel, wind_vector, tnh
    features = (dem,fuel,wind, tnh )
    # label은 sstdf
    label = fire_state
    return features, label

# 데이터셋을 features와 labels로 변환
train_dataset = train_dataset.map(split_features_label)


# # 하나의 배치를 가져와서 시각화
# for (inputs) in train_dataset.take(1):
#     (dem,fuel,wind, tnh), label= inputs
#     print("Terrain shape:", dem.shape)
#     print("Fire seq shape:", fuel.shape)
#     print("Weather seq shape:", wind.shape)
#     print("Weather seq shape:", tnh.shape)
#     plt.figure(figsize=(20,5))
#     plt.subplot(1,4,1)
#     plt.imshow(dem[0])
#     print(tf.reduce_max(dem[0]))
#     print(tf.reduce_min(dem[0]))
#     plt.title("Terrain")
#     plt.axis("off")
#     plt.subplot(1,4,2)
#     plt.imshow(fuel[0].numpy().squeeze(), cmap='gray')
#     print(tf.reduce_max(fuel[0]))
#     print(tf.reduce_min(fuel[0]))
#     plt.title("Fire 0h")
#     plt.axis("off")
#     plt.title("Terrain")
#     plt.axis("off")
#     plt.subplot(1,4,2)
#     plt.imshow(label[0].numpy().squeeze(), cmap='gray')
#     print(tf.reduce_max(label[0]))
#     print(tf.reduce_min(label[0]))
#     plt.title("Fire 0h")
#     plt.axis("off")
#     print(wind[0])
#     print(tnh[0])
    
#     plt.show()
#     break


In [ ]:
model.fit(train_dataset, epochs=300)

In [ ]:
loaded_model = model.load_weights('./cnn_copy_2.h5')

In [ ]:
import matplotlib.pyplot as plt
import random
import math
from tqdm import tqdm

tfrecord_files = ["./ignition/test_128_56ea.tfrecord"]

def split_test_features_label(fire_state, dem, fuel, wind, tnh, ig, w):
    # features는 fire_state, dem, fuel, wind_vector, tnh
    features = (dem, fuel, wind, tnh, ig, w)
    # label은 sstdf
    label = fire_state
    return features, label

# 데이터셋 불러오기
# 1) Test 데이터셋 구성 (batch=16)
# 데이터셋 생성 및 매핑
test_dataset = tf.data.TFRecordDataset(tfrecord_files)
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(1, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)


def split_features_label(fire_state, dem,fuel, wind,tnh, ig,w ):
    # features는 fire_state, dem, fuel, wind_vector, tnh
    features = (dem,fuel,wind, tnh )
    # label은 sstdf
    label = fire_state
    return features, label

# 데이터셋을 features와 labels로 변환
test_dataset = test_dataset.map(split_features_label)

# 2) 리스트로 변환 (테스트셋이 작을 때만)
test_batches = list(test_dataset)
num_batches = len(test_batches)
print(len(test_batches))

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import math
import numpy as np
import time
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.linalg import sqrtm
from tqdm import tqdm

# 데이터셋 파일 경로
#tfrecord_files = ["./test_mid_128_all.tfrecord"]
tfrecord_files = ["./test_last_128_53ea.tfrecord"]
# 데이터셋 불러오기
# 1) Test 데이터셋 구성 (batch=16)
# 데이터셋 생성 및 매핑
test_dataset = tf.data.TFRecordDataset(tfrecord_files)
test_dataset = test_dataset.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(1, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val

def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val

def split_features_label(fire_state, dem,fuel, wind,tnh, ig,w ):
    # features는 fire_state, dem, fuel, wind_vector, tnh
    features = (dem,fuel,wind, tnh )
    # label은 sstdf
    label = fire_state
    return features, label


def calculate_iou(y_true, y_pred, threshold=0.1):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou

def expand_to_bhwc1(x):
    """
    x: (H,W) or (H,W,1)
    return: (1,H,W,1)
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 2:
        x = tf.expand_dims(x, axis=-1)      # (H,W)->(H,W,1)
    x = tf.expand_dims(x, axis=0)           # ->(1,H,W,1)
    return x


def _to_hw(x):
    """
    (B,H,W,1) or (H,W,1) or (H,W) -> (H,W) float32
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 4:  # (B,H,W,1)
        x = x[0, ..., 0]
    elif len(x.shape) == 3:  # (H,W,1)
        x = x[..., 0]
    return x  # (H,W)

def time_to_normalized_pixel_threshold(hour):
    original_pixel_value = -16.25 * hour + 255
    normalized_threshold = original_pixel_value / 255.0
    return normalized_threshold

def create_spread_mask(image, threshold):
    mask = np.where(image >= threshold, image, 0)
    return mask


# 데이터셋을 features와 labels로 변환
test_dataset = test_dataset.map(split_features_label)

# 2) 리스트로 변환 (테스트셋이 작을 때만)
test_batches = list(test_dataset)
num_batches = len(test_batches)
print(len(test_batches))


# 데이터셋 전체를 리스트로 변환 (메모리에 로드)
test_batches = list(test_dataset)
num_batches = len(test_batches)
print(f"데이터셋 로딩 완료. 전체 샘플 수: {num_batches}")


# -------------------------------------------------------------------
# 2. 평가 및 시각화를 위한 헬퍼 함수
# -------------------------------------------------------------------

def calculate_fid(model, real_images, generated_images):
    def preprocess_images(images):
        images_resized = tf.image.resize(images, [75, 75])
        images_rgb = tf.image.grayscale_to_rgb(images_resized)
        images_preprocessed = preprocess_input(images_rgb)
        return images_preprocessed
    real_processed = preprocess_images(real_images)
    gen_processed = preprocess_images(generated_images)
    real_features = model.predict(real_processed)
    gen_features = model.predict(gen_processed)
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)
    ssdiff = np.sum((mu_real - mu_gen)**2.0)
    covmean = sqrtm(sigma_real.dot(sigma_gen))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma_real + sigma_gen - 2.0 * covmean)
    return fid

def calculate_iou_from_mask(y_true, y_pred, threshold=0.1):
    """
    임계값을 기준으로 이진화한 후 IoU를 계산합니다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7) # 0으로 나누는 것을 방지
    return iou.numpy().item()

def time_to_normalized_pixel_threshold(hour):
    original_pixel_value = -16.25 * hour + 255
    normalized_threshold = original_pixel_value / 255.0
    return normalized_threshold

def create_spread_mask(image, threshold):
    mask = np.where(image >= threshold, image, 0)
    return mask


# -------------------------------------------------------------------
# 3. 메인 실행 코드
# -------------------------------------------------------------------

# --- 설정 (이 부분만 필요에 맞게 수정하세요) ---
N = 0  # 시각화를 시작할 그룹 번호 (0: 첫 28개, 1: 다음 28개, ...)
NUM_SAMPLES_TO_VISUALIZE = 28 # 한 번에 시각화할 샘플의 총 개수

# !!! 중요: 학습된 AE 모델을 여기에 로드하세요 !!!
# 예시: model = tf.keras.models.load_model('my_autoencoder_model.h5')
model = tf.keras.models.load_model('./cnn_copy_2.h5')

# FID 계산을 위해 사전 학습된 InceptionV3 모델 로드
inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(75, 75, 3))

# FID 계산에 사용할 전체 이미지 마스크를 저장할 리스트 초기화
all_gt_masks_ae = []
all_pred_masks_ae = []

# 시각화를 시작할 샘플의 인덱스 계산
start_idx = N * NUM_SAMPLES_TO_VISUALIZE

# 인덱스 유효성 검사
if start_idx >= num_batches:
    print(f"오류: 시작 인덱스({start_idx})가 전체 데이터 수({num_batches})를 초과합니다. N 값을 줄여주세요.")
else:
    # 마지막 그룹의 샘플 수가 부족할 경우 실제 시각화할 개수 조정
    num_samples = min(NUM_SAMPLES_TO_VISUALIZE, num_batches - start_idx)

    time_intervals = [4, 8, 12]
    num_time_steps = len(time_intervals)

    num_cols_per_sample_row = num_time_steps
    num_rows_per_sample = 2
    total_plot_cols = num_cols_per_sample_row
    total_plot_rows = num_samples * num_rows_per_sample

    plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5))
    print(f"\n총 {num_samples}개 샘플을 시간대별로 시각화합니다. (인덱스 {start_idx}부터)")
    print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

    for i in range(num_samples):
        current_idx = start_idx + i
        batch_features, batch_label = test_batches[current_idx]

        start_time = time.time()
        # 모델 입력 형식에 맞게 batch_features 전달
        pred_img_raw = model.predict(batch_features)[0]
        prediction_time = time.time() - start_time
        
        pred_img = pred_img_raw
        # batch_label은 (1, H, W, C) 형태일 것이므로 [0]으로 첫번째 샘플 추출
        actual_img = batch_label.numpy()[0]
        
        print(f"\n--- Processing Sample Index: {current_idx} (Prediction Time: {prediction_time:.2f}s) ---")

        current_sample_start_row = i * num_rows_per_sample

        for t, hour in enumerate(time_intervals):
            thr_t = time_to_normalized_pixel_threshold(hour)
            actual_mask = create_spread_mask(actual_img, thr_t)  # (H,W) or (H,W,1)
            pred_mask   = create_spread_mask(pred_img,   thr_t)
            actual_mask_b = expand_to_bhwc1(actual_mask)
            pred_mask_b   = expand_to_bhwc1(pred_mask)

            all_gt_masks_ae.append(actual_mask)
            all_pred_masks_ae.append(pred_mask)

            actual_mask_tensor = tf.convert_to_tensor(actual_mask, dtype=tf.float32)
            pred_mask_tensor = tf.convert_to_tensor(pred_mask, dtype=tf.float32)
            
            union_mask = get_mask(
                actual_mask_b,
                pred_mask_b,
                threshold=0.1,
                mode="union"
            )

            mse_t = masked_mse(actual_mask_b, pred_mask_b, union_mask).numpy()
            ssim_t = masked_ssim(actual_mask_b, pred_mask_b, union_mask, max_val=1.0).numpy()
            iou_t = calculate_iou(actual_mask_b, pred_mask_b, threshold=0.1).numpy()

            # mse_score = tf.keras.losses.MeanSquaredError()(actual_mask_tensor, pred_mask_tensor).numpy().item()
            # ssim_score = tf.image.ssim(
            #     tf.reshape(actual_mask_tensor, [1, 128, 128, 1]),
            #     tf.reshape(pred_mask_tensor, [1, 128, 128, 1]),
            #     max_val=1.0
            # ).numpy().item()
            # iou_score = calculate_iou_from_mask(actual_mask_tensor, pred_mask_tensor)
            
            plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
            plt.imshow(actual_mask, cmap='hot', vmin=0, vmax=1)
            plt.title(f"True {hour}h (Idx:{current_idx})")
            plt.axis('off')

            plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
            plt.imshow(pred_mask, cmap='hot', vmin=0, vmax=1)
            plt.title(f"Pred {hour}h (Idx:{current_idx})\n"
                      f"(MSE: {mse_t:.4f}, SSIM: {ssim_t:.4f}, IoU: {iou_t:.4f})")
            plt.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

########################################
# 1. 공통 유틸 (GAN과 동일하게 맞춤)
########################################

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val


def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val


def calculate_iou(y_true, y_pred, threshold=0.1):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou


def expand_to_bhwc1(x):
    """
    x: (H,W) or (H,W,1)
    return: (1,H,W,1)
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 2:
        x = tf.expand_dims(x, axis=-1)      # (H,W)->(H,W,1)
    x = tf.expand_dims(x, axis=0)           # ->(1,H,W,1)
    return x


def _to_hw(x):
    """
    (B,H,W,1) or (H,W,1) or (H,W) -> (H,W) float32
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 4:  # (B,H,W,1)
        x = x[0, ..., 0]
    elif len(x.shape) == 3:  # (H,W,1)
        x = x[..., 0]
    return x  # (H,W)

def time_to_normalized_pixel_threshold(hour):
    original_pixel_value = -16.25 * hour + 255
    normalized_threshold = original_pixel_value / 255.0
    return normalized_threshold

def create_spread_mask(image, threshold):
    mask = np.where(image >= threshold, image, 0)
    return mask
def get_fire_area_binary(arrival_map, threshold=0.1):
    """
    arrival_map: (H,W), 값은 '불 도달 시간(정규화)' 또는 0
    threshold 기준을 넘은 픽셀을 '불이 번진 영역'으로 취급.
    (CGAN 평가와 동일하게 thr_mask 논리 재사용)
    """
    arrival_map = tf.convert_to_tensor(arrival_map, dtype=tf.float32)
    area_bin = tf.cast(arrival_map > threshold, tf.float32)  # (H,W) 0/1
    return area_bin


def get_fireline_edge(area_bin):
    """
    area_bin: (H,W) 0/1
    return: edge_bin (H,W) 0/1
    """
    x = tf.expand_dims(area_bin, axis=0)    # (1,H,W)
    x = tf.expand_dims(x, axis=-1)          # (1,H,W,1)

    dil = tf.nn.max_pool(
        x,
        ksize=[1,3,3,1],
        strides=[1,1,1,1],
        padding='SAME'
    )
    ero = -tf.nn.max_pool(
        -x,
        ksize=[1,3,3,1],
        strides=[1,1,1,1],
        padding='SAME'
    )

    edge = tf.clip_by_value(dil - ero, 0.0, 1.0)  # (1,H,W,1)
    edge = tf.squeeze(edge, axis=0)               # (H,W,1)
    edge = tf.squeeze(edge, axis=-1)              # (H,W)
    edge_bin = tf.cast(edge > 0.5, tf.float32)
    return edge_bin


def fireline_length(edge_binary, pixel_size=1.0):
    edge_binary = tf.convert_to_tensor(edge_binary, dtype=tf.float32)
    return tf.reduce_sum(edge_binary) * pixel_size


def boundary_iou(edge_true, edge_pred):
    edge_true = tf.convert_to_tensor(edge_true, dtype=tf.float32)
    edge_pred = tf.convert_to_tensor(edge_pred, dtype=tf.float32)

    inter = tf.reduce_sum(edge_true * edge_pred)
    union = tf.reduce_sum(edge_true) + tf.reduce_sum(edge_pred) - inter

    biou = (inter + 1e-7) / (union + 1e-7)
    return biou


def chamfer_distance(edge_true, edge_pred):
    edge_true = tf.convert_to_tensor(edge_true, dtype=tf.float32)
    edge_pred = tf.convert_to_tensor(edge_pred, dtype=tf.float32)

    coords_true = tf.where(edge_true > 0.5)  # (N1,2) [y,x]
    coords_pred = tf.where(edge_pred > 0.5)  # (N2,2)

    cond_empty = tf.logical_or(
        tf.equal(tf.shape(coords_true)[0], 0),
        tf.equal(tf.shape(coords_pred)[0], 0)
    )

    def _nan():
        return tf.constant(np.nan, dtype=tf.float32)

    def _compute():
        diff = tf.cast(
            tf.expand_dims(coords_true, 1) - tf.expand_dims(coords_pred, 0),
            tf.float32
        )  # (N1,N2,2)

        dist = tf.sqrt(tf.reduce_sum(tf.square(diff), axis=2))  # (N1,N2)

        d1 = tf.reduce_mean(tf.reduce_min(dist, axis=1))  # true -> pred
        d2 = tf.reduce_mean(tf.reduce_min(dist, axis=0))  # pred -> true

        return (d1 + d2) / 2.0

    return tf.cond(cond_empty, _nan, _compute)


########################################
# 2. CNN 평가 함수 (GAN과 동일한 메트릭 포함)
########################################

def evaluate_cnn_model(model, dataset, thr_mask=0.1, pixel_size=1.0):
    """
    CNN 모델의 평균 지표를 시간대별(4h,8h,12h)로 계산.
    계산 지표:
      - MSE (masked)
      - SSIM (masked)
      - IoU
      - PerimeterRelErr (화선 둘레 복잡도 상대오차)
      - BoundaryIoU (화선 경계 IoU)
      - ChamferDist (화선 평균 거리 차이)

    반환 형식은 GAN evaluate_model()과 동일한 키를 갖는 dict:
      {
        "MSE": [...],
        "SSIM": [...],
        "IoU": [...],
        "PerimeterRelErr": [...],
        "BoundaryIoU": [...],
        "ChamferDist": [...]
      }
    각 리스트는 [4h 평균, 8h 평균, 12h 평균]
    """

    all_mse_scores   = [[] for _ in range(3)]
    all_ssim_scores  = [[] for _ in range(3)]
    all_iou_scores   = [[] for _ in range(3)]

    all_perim_relerr = [[] for _ in range(3)]
    all_biou_scores  = [[] for _ in range(3)]
    all_chamfer      = [[] for _ in range(3)]

    time_intervals = [4, 8, 12]

    for batch in tqdm(dataset, desc="Evaluating CNN Model"):
        batch_features, batch_label = batch  # batch_label: 실제 12h 번짐맵 or 전체 time map
                                            # (네 파이프라인 기준: batch_size=1 가정)

        # CNN 예측 (12h 시점 예측 결과 한 장)
        pred_img_12h = model.predict(batch_features, verbose=0)[0]   # (H,W,1) or (H,W)
        actual_img_12h = batch_label.numpy()[0]                       # same spatial shape

        for t, hour in enumerate(time_intervals):
            # 1) "해당 시각까지 번진 영역"만 남기는 마스크 생성
            thr_t = time_to_normalized_pixel_threshold(hour)
            actual_mask = create_spread_mask(actual_img_12h, thr_t)  # (H,W) or (H,W,1)
            pred_mask   = create_spread_mask(pred_img_12h,   thr_t)

            # (1,H,W,1)로 맞춤
            actual_mask_b = expand_to_bhwc1(actual_mask)
            pred_mask_b   = expand_to_bhwc1(pred_mask)

            # -----------------------------------------------------------------
            # A. 불 번짐 영역 기반 지표 (MSE / SSIM / IoU)
            # -----------------------------------------------------------------
            union_mask = get_mask(
                actual_mask_b,
                pred_mask_b,
                threshold=thr_mask,
                mode="union"
            )

            mse_t = masked_mse(actual_mask_b, pred_mask_b, union_mask).numpy()
            ssim_t = masked_ssim(actual_mask_b, pred_mask_b, union_mask, max_val=1.0).numpy()
            iou_t = calculate_iou(actual_mask_b, pred_mask_b, threshold=thr_mask).numpy()

            if not np.isnan(mse_t):
                all_mse_scores[t].append(float(mse_t))
            if not np.isnan(ssim_t):
                all_ssim_scores[t].append(float(ssim_t))
            all_iou_scores[t].append(float(iou_t))

            # -----------------------------------------------------------------
            # B. 화선(경계선) 기반 지표 (PerimeterRelErr / BoundaryIoU / ChamferDist)
            # -----------------------------------------------------------------
            # 이진 번짐 영역 (H,W)
            area_true_bin = get_fire_area_binary(_to_hw(actual_mask_b), threshold=thr_mask)
            area_pred_bin = get_fire_area_binary(_to_hw(pred_mask_b),   threshold=thr_mask)

            # 경계선 추출
            edge_true = get_fireline_edge(area_true_bin)  # (H,W)
            edge_pred = get_fireline_edge(area_pred_bin)  # (H,W)

            # 화선 길이 비교 (Perimeter Relative Error)
            L_true = fireline_length(edge_true, pixel_size=pixel_size)
            L_pred = fireline_length(edge_pred, pixel_size=pixel_size)

            perim_relerr_t = np.nan
            if float(L_true.numpy()) > 0.0:
                perim_relerr_t = tf.abs(L_true - L_pred) / (L_true + 1e-7)
                perim_relerr_t = float(perim_relerr_t.numpy())

            if not np.isnan(perim_relerr_t):
                all_perim_relerr[t].append(perim_relerr_t)

            # Boundary IoU (화선 위치 일치 정도)
            biou_t = boundary_iou(edge_true, edge_pred).numpy()
            all_biou_scores[t].append(float(biou_t))

            # Chamfer Distance (화선 평균 거리 오차)
            chamfer_t = chamfer_distance(edge_true, edge_pred).numpy()
            if not np.isnan(chamfer_t):
                all_chamfer[t].append(float(chamfer_t))

    # 시간대별 평균 계산
    avg_mse   = [np.mean(s) if len(s)>0 else np.nan for s in all_mse_scores]
    avg_ssim  = [np.mean(s) if len(s)>0 else np.nan for s in all_ssim_scores]
    avg_iou   = [np.mean(s) if len(s)>0 else np.nan for s in all_iou_scores]

    avg_perim = [np.mean(s) if len(s)>0 else np.nan for s in all_perim_relerr]
    avg_biou  = [np.mean(s) if len(s)>0 else np.nan for s in all_biou_scores]
    avg_chamf = [np.mean(s) if len(s)>0 else np.nan for s in all_chamfer]

    return {
        "MSE": avg_mse,               # 낮을수록 좋음 (도달시간 차이 작음)
        "SSIM": avg_ssim,             # 높을수록 좋음 (형태/패턴 유사)
        "IoU": avg_iou,               # 높을수록 좋음 (번진 면적 일치)
        "PerimeterRelErr": avg_perim, # 낮을수록 좋음 (화선 복잡도 잘 재현)
        "BoundaryIoU": avg_biou,      # 높을수록 좋음 (화선 위치 일치)
        "ChamferDist": avg_chamf      # 낮을수록 좋음 (화선 경계 평균 오차 작음)
    }

# 데이터셋 파일 경로
tfrecord_files = ["./train_mid_128_all.tfrecord"]
#tfrecord_files = ["./ignition/test_128_56ea.tfrecord"]
# 데이터셋 불러오기
# 1) Test 데이터셋 구성 (batch=16)
# 데이터셋 생성 및 매핑
train_dataset = tf.data.TFRecordDataset(tfrecord_files)
train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(1, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.map(split_features_label)
# 데이터셋 파일 경로
tfrecord_files = ["./test_mid_128_all.tfrecord"]
#tfrecord_files = ["./ignition/test_128_56ea.tfrecord"]
# 데이터셋 불러오기
# 1) Test 데이터셋 구성 (batch=16)
# 데이터셋 생성 및 매핑
test_dataset = tf.data.TFRecordDataset(tfrecord_files)
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(1, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(split_features_label)

########################################
# 3. 실행 & 시각화
########################################

print("\nEvaluating CNN on Test Dataset...")
# test_dataset는 (features, label) 형태여야 하고
# label은 실제 12h map, features는 모델 입력인 상태라고 가정
cnn_test_results = evaluate_cnn_model(model, test_dataset, thr_mask=0.1, pixel_size=1.0)

print("\nEvaluating CNN on Train Dataset...")
# train_dataset도 동일한 구조여야 함 (split_features_label 이후)
cnn_train_results = evaluate_cnn_model(model, train_dataset, thr_mask=0.1, pixel_size=1.0)

print("\n--- CNN Evaluation Results ---")
print("Train:", cnn_train_results)
print("Test :", cnn_test_results)


def plot_results(train_res, test_res, model_name="CNN"):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    fig, axes = plt.subplots(2, 3, figsize=(28, 12))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=18)

    metric_names = [
        ("MSE", "Mean Squared Error (↓ better)"),
        ("SSIM", "Structural Similarity (↑ better)"),
        ("IoU", "Burned Area IoU (↑ better)"),
        ("PerimeterRelErr", "Perimeter Relative Error (↓ better)"),
        ("BoundaryIoU", "Boundary IoU (↑ better)"),
        ("ChamferDist", "Chamfer Distance (px, ↓ better)")
    ]

    for ax, (key, title) in zip(axes.flatten(), metric_names):
        train_vals = train_res[key]
        test_vals  = test_res[key]

        rects1 = ax.bar(x - width/2, train_vals, width, label='Train')
        rects2 = ax.bar(x + width/2, test_vals,  width, label='Test')

        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.legend()

        ax.bar_label(rects1, padding=3, fmt='%.4f')
        ax.bar_label(rects2, padding=3, fmt='%.4f')

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


print("\nPlotting CNN results...")
plot_results(cnn_train_results, cnn_test_results, model_name="CNN")


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import math
import numpy as np
import time
# --- 설정 (이 부분만 수정하세요) ---

# 시각화를 시작할 그룹 번호 (0부터 시작)
N = 1
# 한 번에 시각화할 샘플의 총 개수
NUM_SAMPLES_TO_VISUALIZE = 28
# 시각화 결과에서 한 행에 표시할 샘플의 개수 (GAN과 동일하게 1로 고정하여 한 줄에 한 샘플만)
SAMPLES_PER_ROW = 1


# --- Helper 함수 (수정 없음) ---
def time_to_normalized_pixel_threshold(hour):
    original_pixel_value = -16.25 * hour + 255
    normalized_threshold = original_pixel_value / 255.0
    return normalized_threshold

def create_spread_mask(image, threshold):
    mask = np.where(image >= threshold, image, 0) # '=' 포함하여 경계 픽셀도 포함
    return mask

# --- 메인 시각화 루프 (수정됨) ---

# 시각화를 시작할 샘플의 인덱스 계산
start_idx = N * NUM_SAMPLES_TO_VISUALIZE

if start_idx >= num_batches:
    print(f"오류: 시작 인덱스({start_idx})가 전체 데이터 수({num_batches})를 초과합니다. N 값을 줄여주세요.")
else:
    # 실제로 시각화할 샘플 수 조정
    if start_idx + NUM_SAMPLES_TO_VISUALIZE > num_batches:
        print(f"경고: 마지막 그룹의 샘플 수가 부족합니다. {num_batches - start_idx}개만 시각화합니다.")
        num_samples = num_batches - start_idx
    else:
        num_samples = NUM_SAMPLES_TO_VISUALIZE

    # 시간대별 비교를 위한 설정
    time_intervals = [4, 8, 12]
    num_time_steps = len(time_intervals) # 3

    # Matplotlib 서브플롯 레이아웃 계산 (GAN과 동일한 구조)
    num_cols_per_sample_row = num_time_steps  # 3 (4h, 8h, 12h)
    num_rows_per_sample = 2                   # 2 (실제, 예측)
    
    total_plot_cols = num_cols_per_sample_row
    total_plot_rows = num_samples * num_rows_per_sample

    plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5))
    print(f"\n총 {num_samples}개 샘플을 시간대별로 시각화합니다. (인덱스 {start_idx}부터)")
    print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

    for i in range(num_samples):
        # 현재 샘플의 실제 인덱스
        current_idx = start_idx + i
        
        # 데이터 가져오기
        batch_features, batch_label = test_batches[current_idx]

        # 모델 예측 수행 및 시간 측정
        start_time = time.time()
        pred_img_raw = model.predict(batch_features)[0]
        prediction_time = time.time() - start_time
        
        # [0, 1] 범위로 정규화 (모델 출력에 따라 조정)
        pred_img = pred_img_raw
        actual_img = batch_label.numpy()[0]
        
        print(f"\n--- Processing Sample Index: {current_idx} (Prediction Time: {prediction_time:.2f}s) ---")

        # 각 샘플에 대한 서브플롯 시작 행 위치 계산
        current_sample_start_row = i * num_rows_per_sample

        for t, hour in enumerate(time_intervals):
            threshold = time_to_normalized_pixel_threshold(hour)
            
            # 후처리를 통해 시간대별 마스크 생성
            pred_mask = create_spread_mask(pred_img, threshold)
            actual_mask = create_spread_mask(actual_img, threshold)

            # --- 점수 계산 ---
            actual_mask_tensor = tf.convert_to_tensor(actual_mask, dtype=tf.float32)
            pred_mask_tensor = tf.convert_to_tensor(pred_mask, dtype=tf.float32)
            
            # MSE
            mse_loss = tf.keras.losses.MeanSquaredError()
            mse_score = mse_loss(actual_mask_tensor, pred_mask_tensor).numpy().item()

            # SSIM
            ssim_score = tf.image.ssim(
                tf.expand_dims(actual_mask_tensor, axis=0), 
                tf.expand_dims(pred_mask_tensor, axis=0), 
                max_val=1.0
            ).numpy().item()
            
            # --- 시각화 ---
            
            # 실제 이미지 플롯 (현재 샘플의 첫 번째 행)
            plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
            plt.imshow(actual_mask, cmap='hot', vmin=0, vmax=1)
            plt.title(f"True {hour}h (Idx:{current_idx})")
            plt.axis('off')

            # 예측 이미지 플롯 (현재 샘플의 두 번째 행)
            plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
            plt.imshow(pred_mask, cmap='hot', vmin=0, vmax=1)
            plt.title(f"Pred {hour}h (Idx:{current_idx})\n"
                      f"(MSE: {mse_score:.4f}, SSIM: {ssim_score:.4f})")
            plt.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()



In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

########################################
# 1. 공통 유틸 (GAN과 동일하게 맞춤)
########################################

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val


def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val


def calculate_iou(y_true, y_pred, threshold=0.1):
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou


def expand_to_bhwc1(x):
    """
    x: (H,W) or (H,W,1)
    return: (1,H,W,1)
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 2:
        x = tf.expand_dims(x, axis=-1)      # (H,W)->(H,W,1)
    x = tf.expand_dims(x, axis=0)           # ->(1,H,W,1)
    return x


def _to_hw(x):
    """
    (B,H,W,1) or (H,W,1) or (H,W) -> (H,W) float32
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 4:  # (B,H,W,1)
        x = x[0, ..., 0]
    elif len(x.shape) == 3:  # (H,W,1)
        x = x[..., 0]
    return x  # (H,W)


def get_fire_area_binary(arrival_map, threshold=0.1):
    """
    arrival_map: (H,W), 값은 '불 도달 시간(정규화)' 또는 0
    threshold 기준을 넘은 픽셀을 '불이 번진 영역'으로 취급.
    (CGAN 평가와 동일하게 thr_mask 논리 재사용)
    """
    arrival_map = tf.convert_to_tensor(arrival_map, dtype=tf.float32)
    area_bin = tf.cast(arrival_map > threshold, tf.float32)  # (H,W) 0/1
    return area_bin


def get_fireline_edge(area_bin):
    """
    area_bin: (H,W) 0/1
    return: edge_bin (H,W) 0/1
    """
    x = tf.expand_dims(area_bin, axis=0)    # (1,H,W)
    x = tf.expand_dims(x, axis=-1)          # (1,H,W,1)

    dil = tf.nn.max_pool(
        x,
        ksize=[1,3,3,1],
        strides=[1,1,1,1],
        padding='SAME'
    )
    ero = -tf.nn.max_pool(
        -x,
        ksize=[1,3,3,1],
        strides=[1,1,1,1],
        padding='SAME'
    )

    edge = tf.clip_by_value(dil - ero, 0.0, 1.0)  # (1,H,W,1)
    edge = tf.squeeze(edge, axis=0)               # (H,W,1)
    edge = tf.squeeze(edge, axis=-1)              # (H,W)
    edge_bin = tf.cast(edge > 0.5, tf.float32)
    return edge_bin


def fireline_length(edge_binary, pixel_size=1.0):
    edge_binary = tf.convert_to_tensor(edge_binary, dtype=tf.float32)
    return tf.reduce_sum(edge_binary) * pixel_size


def boundary_iou(edge_true, edge_pred):
    edge_true = tf.convert_to_tensor(edge_true, dtype=tf.float32)
    edge_pred = tf.convert_to_tensor(edge_pred, dtype=tf.float32)

    inter = tf.reduce_sum(edge_true * edge_pred)
    union = tf.reduce_sum(edge_true) + tf.reduce_sum(edge_pred) - inter

    biou = (inter + 1e-7) / (union + 1e-7)
    return biou


def chamfer_distance(edge_true, edge_pred):
    edge_true = tf.convert_to_tensor(edge_true, dtype=tf.float32)
    edge_pred = tf.convert_to_tensor(edge_pred, dtype=tf.float32)

    coords_true = tf.where(edge_true > 0.5)  # (N1,2) [y,x]
    coords_pred = tf.where(edge_pred > 0.5)  # (N2,2)

    cond_empty = tf.logical_or(
        tf.equal(tf.shape(coords_true)[0], 0),
        tf.equal(tf.shape(coords_pred)[0], 0)
    )

    def _nan():
        return tf.constant(np.nan, dtype=tf.float32)

    def _compute():
        diff = tf.cast(
            tf.expand_dims(coords_true, 1) - tf.expand_dims(coords_pred, 0),
            tf.float32
        )  # (N1,N2,2)

        dist = tf.sqrt(tf.reduce_sum(tf.square(diff), axis=2))  # (N1,N2)

        d1 = tf.reduce_mean(tf.reduce_min(dist, axis=1))  # true -> pred
        d2 = tf.reduce_mean(tf.reduce_min(dist, axis=0))  # pred -> true

        return (d1 + d2) / 2.0

    return tf.cond(cond_empty, _nan, _compute)


########################################
# 2. CNN 평가 함수 (GAN과 동일한 메트릭 포함)
########################################

def evaluate_cnn_model(model, dataset, thr_mask=0.1, pixel_size=1.0):
    """
    CNN 모델의 평균 지표를 시간대별(4h,8h,12h)로 계산.
    계산 지표:
      - MSE (masked)
      - SSIM (masked)
      - IoU
      - PerimeterRelErr (화선 둘레 복잡도 상대오차)
      - BoundaryIoU (화선 경계 IoU)
      - ChamferDist (화선 평균 거리 차이)

    반환 형식은 GAN evaluate_model()과 동일한 키를 갖는 dict:
      {
        "MSE": [...],
        "SSIM": [...],
        "IoU": [...],
        "PerimeterRelErr": [...],
        "BoundaryIoU": [...],
        "ChamferDist": [...]
      }
    각 리스트는 [4h 평균, 8h 평균, 12h 평균]
    """

    all_mse_scores   = [[] for _ in range(3)]
    all_ssim_scores  = [[] for _ in range(3)]
    all_iou_scores   = [[] for _ in range(3)]

    all_perim_relerr = [[] for _ in range(3)]
    all_biou_scores  = [[] for _ in range(3)]
    all_chamfer      = [[] for _ in range(3)]

    time_intervals = [4, 8, 12]

    for batch in tqdm(dataset, desc="Evaluating CNN Model"):
        batch_features, batch_label = batch  # batch_label: 실제 12h 번짐맵 or 전체 time map
                                            # (네 파이프라인 기준: batch_size=1 가정)

        # CNN 예측 (12h 시점 예측 결과 한 장)
        pred_img_12h = model.predict(batch_features, verbose=0)[0]   # (H,W,1) or (H,W)
        actual_img_12h = batch_label.numpy()[0]                       # same spatial shape

        for t, hour in enumerate(time_intervals):
            # 1) "해당 시각까지 번진 영역"만 남기는 마스크 생성
            thr_t = time_to_normalized_pixel_threshold(hour)
            actual_mask = create_spread_mask(actual_img_12h, thr_t)  # (H,W) or (H,W,1)
            pred_mask   = create_spread_mask(pred_img_12h,   thr_t)

            # (1,H,W,1)로 맞춤
            actual_mask_b = expand_to_bhwc1(actual_mask)
            pred_mask_b   = expand_to_bhwc1(pred_mask)

            # -----------------------------------------------------------------
            # A. 불 번짐 영역 기반 지표 (MSE / SSIM / IoU)
            # -----------------------------------------------------------------
            union_mask = get_mask(
                actual_mask_b,
                pred_mask_b,
                threshold=thr_mask,
                mode="union"
            )

            mse_t = masked_mse(actual_mask_b, pred_mask_b, union_mask).numpy()
            ssim_t = masked_ssim(actual_mask_b, pred_mask_b, union_mask, max_val=1.0).numpy()
            iou_t = calculate_iou(actual_mask_b, pred_mask_b, threshold=thr_mask).numpy()

            if not np.isnan(mse_t):
                all_mse_scores[t].append(float(mse_t))
            if not np.isnan(ssim_t):
                all_ssim_scores[t].append(float(ssim_t))
            all_iou_scores[t].append(float(iou_t))

            # -----------------------------------------------------------------
            # B. 화선(경계선) 기반 지표 (PerimeterRelErr / BoundaryIoU / ChamferDist)
            # -----------------------------------------------------------------
            # 이진 번짐 영역 (H,W)
            area_true_bin = get_fire_area_binary(_to_hw(actual_mask_b), threshold=thr_mask)
            area_pred_bin = get_fire_area_binary(_to_hw(pred_mask_b),   threshold=thr_mask)

            # 경계선 추출
            edge_true = get_fireline_edge(area_true_bin)  # (H,W)
            edge_pred = get_fireline_edge(area_pred_bin)  # (H,W)

            # 화선 길이 비교 (Perimeter Relative Error)
            L_true = fireline_length(edge_true, pixel_size=pixel_size)
            L_pred = fireline_length(edge_pred, pixel_size=pixel_size)

            perim_relerr_t = np.nan
            if float(L_true.numpy()) > 0.0:
                perim_relerr_t = tf.abs(L_true - L_pred) / (L_true + 1e-7)
                perim_relerr_t = float(perim_relerr_t.numpy())

            if not np.isnan(perim_relerr_t):
                all_perim_relerr[t].append(perim_relerr_t)

            # Boundary IoU (화선 위치 일치 정도)
            biou_t = boundary_iou(edge_true, edge_pred).numpy()
            all_biou_scores[t].append(float(biou_t))

            # Chamfer Distance (화선 평균 거리 오차)
            chamfer_t = chamfer_distance(edge_true, edge_pred).numpy()
            if not np.isnan(chamfer_t):
                all_chamfer[t].append(float(chamfer_t))

    # 시간대별 평균 계산
    avg_mse   = [np.mean(s) if len(s)>0 else np.nan for s in all_mse_scores]
    avg_ssim  = [np.mean(s) if len(s)>0 else np.nan for s in all_ssim_scores]
    avg_iou   = [np.mean(s) if len(s)>0 else np.nan for s in all_iou_scores]

    avg_perim = [np.mean(s) if len(s)>0 else np.nan for s in all_perim_relerr]
    avg_biou  = [np.mean(s) if len(s)>0 else np.nan for s in all_biou_scores]
    avg_chamf = [np.mean(s) if len(s)>0 else np.nan for s in all_chamfer]

    return {
        "MSE": avg_mse,               # 낮을수록 좋음 (도달시간 차이 작음)
        "SSIM": avg_ssim,             # 높을수록 좋음 (형태/패턴 유사)
        "IoU": avg_iou,               # 높을수록 좋음 (번진 면적 일치)
        "PerimeterRelErr": avg_perim, # 낮을수록 좋음 (화선 복잡도 잘 재현)
        "BoundaryIoU": avg_biou,      # 높을수록 좋음 (화선 위치 일치)
        "ChamferDist": avg_chamf      # 낮을수록 좋음 (화선 경계 평균 오차 작음)
    }


########################################
# 3. 실행 & 시각화
########################################

print("\nEvaluating CNN on Test Dataset...")
# test_dataset는 (features, label) 형태여야 하고
# label은 실제 12h map, features는 모델 입력인 상태라고 가정
cnn_test_results = evaluate_cnn_model(model, test_dataset, thr_mask=0.1, pixel_size=1.0)

print("\nEvaluating CNN on Train Dataset...")
# train_dataset도 동일한 구조여야 함 (split_features_label 이후)
cnn_train_results = evaluate_cnn_model(model, train_dataset, thr_mask=0.1, pixel_size=1.0)

print("\n--- CNN Evaluation Results ---")
print("Train:", cnn_train_results)
print("Test :", cnn_test_results)


def plot_results(train_res, test_res, model_name="CNN"):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    fig, axes = plt.subplots(2, 3, figsize=(28, 12))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=18)

    metric_names = [
        ("MSE", "Mean Squared Error (↓ better)"),
        ("SSIM", "Structural Similarity (↑ better)"),
        ("IoU", "Burned Area IoU (↑ better)"),
        ("PerimeterRelErr", "Perimeter Relative Error (↓ better)"),
        ("BoundaryIoU", "Boundary IoU (↑ better)"),
        ("ChamferDist", "Chamfer Distance (px, ↓ better)")
    ]

    for ax, (key, title) in zip(axes.flatten(), metric_names):
        train_vals = train_res[key]
        test_vals  = test_res[key]

        rects1 = ax.bar(x - width/2, train_vals, width, label='Train')
        rects2 = ax.bar(x + width/2, test_vals,  width, label='Test')

        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.legend()

        ax.bar_label(rects1, padding=3, fmt='%.4f')
        ax.bar_label(rects2, padding=3, fmt='%.4f')

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


print("\nPlotting CNN results...")
plot_results(cnn_train_results, cnn_test_results, model_name="CNN")


In [ ]:
# 모델 평가 실행
print("\nEvaluating on Test Dataset...")
print(f"Test Dataset  -> MSE: {cnn_test_results['MSE']}, SSIM: {cnn_test_results['SSIM']} , IoU: {cnn_test_results['IoU']}")
print("\nEvaluating on Train Dataset...")
print(f"Train Dataset -> MSE: {cnn_train_results['MSE']}, SSIM: {cnn_train_results['SSIM']},, IoU: {cnn_train_results['IoU']}")

In [ ]:
from tqdm import tqdm
def evaluate_cnn_model(model, dataset):
    """CNN 모델의 평균 MSE, SSIM, IoU 점수를 시간대별로 계산합니다."""
    all_mse_scores = [[] for _ in range(3)]
    all_ssim_scores = [[] for _ in range(3)]
    all_iou_scores = [[] for _ in range(3)] # IoU 스코어 리스트 추가
    time_intervals = [4, 8, 12]

    mse_loss_fn = tf.keras.losses.MeanSquaredError()

    for batch in tqdm(dataset, desc="Evaluating CNN Model"):
        batch_features, batch_label = batch
        
        pred_img_12h = model.predict(batch_features, verbose=0)[0]
        pred_img_12h = pred_img_12h
        actual_img_12h = batch_label.numpy()[0]

        for t, hour in enumerate(time_intervals):
            threshold = time_to_normalized_pixel_threshold(hour)
            
            pred_mask = create_spread_mask(pred_img_12h, threshold)
            actual_mask = create_spread_mask(actual_img_12h, threshold)

            pred_mask_tensor = tf.convert_to_tensor(pred_mask, dtype=tf.float32)
            actual_mask_tensor = tf.convert_to_tensor(actual_mask, dtype=tf.float32)
            
            # MSE & SSIM 계산 (기존과 동일)
            mse = mse_loss_fn(actual_mask_tensor, pred_mask_tensor).numpy()
            ssim = tf.image.ssim(
                tf.expand_dims(actual_mask_tensor, axis=0),
                tf.expand_dims(pred_mask_tensor, axis=0),
                max_val=1.0
            ).numpy().mean()

            all_mse_scores[t].append(mse.item())
            all_ssim_scores[t].append(ssim.item())
            
            # --- IoU 계산 추가 ---
            # 마스크는 이미 0이 아닌 값들이 영역을 의미하므로, 0보다 큰지 여부로 이진화
            pred_bin = tf.cast(pred_mask_tensor > 0, dtype=tf.float32)
            target_bin = tf.cast(actual_mask_tensor > 0, dtype=tf.float32)
            
            intersection = tf.reduce_sum(pred_bin * target_bin)
            union = tf.reduce_sum(pred_bin) + tf.reduce_sum(target_bin) - intersection
            
            iou = (intersection + 1e-7) / (union + 1e-7)
            all_iou_scores[t].append(iou.numpy())
            # --------------------

    avg_mse = [np.mean(scores) for scores in all_mse_scores]
    avg_ssim = [np.mean(scores) for scores in all_ssim_scores]
    avg_iou = [np.mean(scores) for scores in all_iou_scores] # IoU 평균 계산 추가

    return {"MSE": avg_mse, "SSIM": avg_ssim, "IoU": avg_iou}

# --- 3. 메인 실행 및 시각화 ---

# 학습된 CNN 모델 로드 (예시)
# cnn_model = tf.keras.models.load_model('path/to/your/cnn_model')

# 데이터셋 경로
# train_tfrecord_path = "./train_cnn_data.tfrecord" # CNN 학습 데이터셋 경로로 변경
# test_tfrecord_path = "./test_cnn_data.tfrecord"

# 데이터셋 로드
# print("Loading CNN datasets...")
# cnn_train_dataset = load_dataset(train_tfrecord_path)
# cnn_test_dataset = load_dataset(test_tfrecord_path)
# print("CNN Datasets loaded.")


tfrecord_test2 = ["./ignition/train_128_56ea.tfrecord"]
test_dataset2 = tf.data.TFRecordDataset(tfrecord_test2)
test_dataset2 = test_dataset2.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset2 = test_dataset2.batch(1, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
test_dataset2 = test_dataset2.prefetch(tf.data.AUTOTUNE)
test_dataset2 = test_dataset2.map(split_features_label)
test_dataset2 = test_dataset2.take(500)
print("\nEvaluating CNN on Train Dataset...")
cnn_train_results = evaluate_cnn_model(model, train_dataset)

# 결과 출력 (예시 값)
# cnn_train_results = {"MSE": [0.0025, 0.0045, 0.0950], "SSIM": [0.9850, 0.9650, 0.5500]}
# cnn_test_results = {"MSE": [0.0029, 0.0051, 0.1150], "SSIM": [0.9810, 0.9600, 0.5200]}
print("\n--- CNN Evaluation Results (Example) ---")
print(f"Train Dataset -> MSE: {cnn_train_results['MSE']}, SSIM: {cnn_train_results['SSIM']}")
print(f"Test Dataset  -> MSE: {cnn_test_results['MSE']}, SSIM: {cnn_test_results['SSIM']}")


# --- 시각화 (GAN 코드와 동일한 함수 사용) ---
def plot_results(train_res, model_name=""):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    # subplots를 1x2에서 1x3으로 변경하고, figsize 너비를 늘립니다.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=16)

    # MSE 그래프 (기존과 동일)
    rects1 = ax1.bar(x - width/2, train_res['MSE'], width, label='Train')
    rects2 = ax1.bar(x + width/2, test_res['MSE'], width, label='Test')
    ax1.set_ylabel('Mean Squared Error (MSE)')
    ax1.set_title('Average MSE (Lower is Better)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.bar_label(rects1, padding=3, fmt='%.4f')
    ax1.bar_label(rects2, padding=3, fmt='%.4f')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # SSIM 그래프 (기존과 동일)
    rects3 = ax2.bar(x - width/2, train_res['SSIM'], width, label='Train')
    rects4 = ax2.bar(x + width/2, test_res['SSIM'], width, label='Test')
    ax2.set_ylabel('Structural Similarity (SSIM)')
    ax2.set_title('Average SSIM (Higher is Better)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.legend()
    ax2.bar_label(rects3, padding=3, fmt='%.4f')
    ax2.bar_label(rects4, padding=3, fmt='%.4f')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    # --- IoU 그래프 추가 ---
    rects5 = ax3.bar(x - width/2, train_res['IoU'], width, label='Train')
    rects6 = ax3.bar(x + width/2, test_res['IoU'], width, label='Test')
    ax3.set_ylabel('Intersection over Union (IoU)')
    ax3.set_title('Average IoU (Higher is Better)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.legend()
    ax3.bar_label(rects5, padding=3, fmt='%.4f')
    ax3.bar_label(rects6, padding=3, fmt='%.4f')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    # --------------------
    
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("\nPlotting CNN results...")
plot_results(cnn_train_results, cnn_test_results, model_name="CNN")

In [ ]:
NN = 0
start_sample_idx = 36*NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 36 # 원하는 샘플 수 (예: 1~36)
if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")
if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit() # 더 이상 진행할 필요가 없을 때 스크립트를 종료
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))
ncols_per_pair = 2 # 실제/예측 한 쌍당 차지하는 subplot 열의 수
num_pairs_per_row = 2 # 한 행에 표시할 쌍의 개수 (원래 12열 중 2열씩 6쌍)
ncols = num_pairs_per_row * ncols_per_pair # 총 열 수 (12)
nrows = math.ceil(num_samples_to_visualize / num_pairs_per_row) # 총 행 수
plt.figure(figsize=(ncols * 2, nrows * 2)) # 가로 24인치, 세로 nrows*2인치 (원래는 가로 12*2=24였는데, 설명을 위해 조정)
print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {nrows} 행 x {ncols} 열")
for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    # 현재 샘플의 절대 인덱스 (test_batches 리스트에서의 인덱스)
    bidx = sequential_batch_idxs[current_sample_relative_idx] 
    
    batch_features, batch_label = test_batches[bidx]

    # 모델의 input() 정의에 맞게 딕셔너리 형태로 전달
    # 현재 `split_test_features_label`은 딕셔너리를 반환하므로 바로 사용할 수 있습니다.
    # 만약 모델이 개별 텐서를 인자로 받는다면 아래처럼 풀어야 합니다.
    # dem, fuel, wind, tnh, ig, w = [feat.numpy() for feat in batch_features]
    # pred_img = model.predict((dem, fuel, wind, tnh))[0].squeeze() / 2 + 0.5
    
    # 모델 예측 (모델이 딕셔너리 입력을 받는다고 가정)
    pred_img = model(batch_features).numpy()[0].squeeze() / 2 + 0.5 # model.predict() 대신 model() 사용 (Eager execution)
    actual_img = batch_label.numpy()[0].squeeze()

    # subplot 인덱스 계산
    # current_sample_relative_idx는 0부터 num_samples_to_visualize-1까지 증가
    row_idx_in_plot = i // num_pairs_per_row
    pair_idx_in_row = i % num_pairs_per_row # 한 행 내에서 몇 번째 쌍인지 (0부터 5)

    # 실제 이미지 subplot
    plt.subplot(nrows, ncols, row_idx_in_plot * ncols + pair_idx_in_row * ncols_per_pair + 1)
    plt.imshow(actual_img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"Actual (Idx {bidx})") # 인덱스 표시
    plt.axis('off')

    # 예측 이미지 subplot
    plt.subplot(nrows, ncols, row_idx_in_plot * ncols + pair_idx_in_row * ncols_per_pair + 2)
    plt.imshow(pred_img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"Pred (Idx {bidx})") # 인덱스 표시
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
import tensorflow as tf

# 테스트 데이터셋에서 샘플 수 설정
num_samples = 30

# test_dataset에서 데이터를 추출하기 위한 반복자 생성
test_iterator = iter(test_dataset)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []
# 랜덤한 배치 인덱스 선택

total_samples_collected = 0  # 수집된 샘플 수 추적

while total_samples_collected < num_samples:
    try:
        # test_dataset에서 하나의 배치를 가져옴
        random_batch_idx = random.randint(0, num_batches - 1)
        test_batch = list(test_dataset.skip(random_batch_idx).take(1))[0]

# 피처와 레이블 분리
        test_features, test_label = test_batch
        test_features_numpy = [feature.numpy() for feature in test_features]
        
        # 배치 내의 각 샘플을 순회하여 처리
        for i in range(test_features_numpy[0].shape[0]):
            if total_samples_collected >= num_samples:
                break

            # noise 생성
            fake_tensor = tf.zeros((1, 128))
            fuel, dem, wind_vector, tnh, ig, w = test_features_numpy


            # generator로 이미지 생성
            generated_image = model.predict([ fuel, dem, wind_vector, tnh])
            generated_image = generated_image[i]
            generated_image =  generated_image
            generated_image = np.squeeze(generated_image, axis=2)

            generated_images.append(generated_image)

            # 실제 이미지 선택
            real_image = test_label[i]
            real_images.append(real_image)

            total_samples_collected += 1

    except StopIteration:
        print("더 이상 가져올 데이터가 없습니다.")
        break

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/3), ncols=6, figsize=(12, 2*int(num_samples/3)))
idx = 0
for i in range(int(num_samples/3)):
    axs[i, 0].imshow(real_images[idx], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx+1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx+1], cmap='hot')
    axs[i, 3].axis('off')
    axs[i, 4].imshow(real_images[idx+2], cmap='hot')
    axs[i, 4].axis('off')
    axs[i, 5].imshow(generated_images[idx+2], cmap='hot')
    axs[i, 5].axis('off')
    idx += 3

# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)
plt.show()

# MSE 및 SSIM 계산 함수
def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
     # Tensor를 NumPy 배열로 변환
    if isinstance(image1, tf.Tensor):
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(image1, np.squeeze(image2, axis=2), data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i], real_images[i]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i], real_images[i]) for i in range(num_samples)]

# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')

In [ ]:
NN = 11
start_sample_idx = 36*NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 36 # 원하는 샘플 수 (예: 1~36)
if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")
if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit() # 더 이상 진행할 필요가 없을 때 스크립트를 종료
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))
ncols_per_pair = 2 # 실제/예측 한 쌍당 차지하는 subplot 열의 수
num_pairs_per_row = 6 # 한 행에 표시할 쌍의 개수 (원래 12열 중 2열씩 6쌍)
ncols = num_pairs_per_row * ncols_per_pair # 총 열 수 (12)
nrows = math.ceil(num_samples_to_visualize / num_pairs_per_row) # 총 행 수
plt.figure(figsize=(ncols * 2, nrows * 2)) # 가로 24인치, 세로 nrows*2인치 (원래는 가로 12*2=24였는데, 설명을 위해 조정)
print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {nrows} 행 x {ncols} 열")
for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    # 현재 샘플의 절대 인덱스 (test_batches 리스트에서의 인덱스)
    bidx = sequential_batch_idxs[current_sample_relative_idx] 
    
    batch_features, batch_label = test_batches[bidx]

    # 모델의 input() 정의에 맞게 딕셔너리 형태로 전달
    # 현재 `split_test_features_label`은 딕셔너리를 반환하므로 바로 사용할 수 있습니다.
    # 만약 모델이 개별 텐서를 인자로 받는다면 아래처럼 풀어야 합니다.
    # dem, fuel, wind, tnh, ig, w = [feat.numpy() for feat in batch_features]
    # pred_img = model.predict((dem, fuel, wind, tnh))[0].squeeze() / 2 + 0.5
    
    # 모델 예측 (모델이 딕셔너리 입력을 받는다고 가정)
    pred_img = model(batch_features).numpy()[0].squeeze() / 2 + 0.5 # model.predict() 대신 model() 사용 (Eager execution)
    actual_img = batch_label.numpy()[0].squeeze()

    # subplot 인덱스 계산
    # current_sample_relative_idx는 0부터 num_samples_to_visualize-1까지 증가
    row_idx_in_plot = i // num_pairs_per_row
    pair_idx_in_row = i % num_pairs_per_row # 한 행 내에서 몇 번째 쌍인지 (0부터 5)

    # 실제 이미지 subplot
    plt.subplot(nrows, ncols, row_idx_in_plot * ncols + pair_idx_in_row * ncols_per_pair + 1)
    plt.imshow(actual_img, cmap='hot', vmin=-1, vmax=1)
    plt.title(f"Actual (Idx {bidx})") # 인덱스 표시
    plt.axis('off')

    # 예측 이미지 subplot
    plt.subplot(nrows, ncols, row_idx_in_plot * ncols + pair_idx_in_row * ncols_per_pair + 2)
    plt.imshow(pred_img, cmap='hot', vmin=0, vmax=1)
    plt.title(f"Pred (Idx {bidx})") # 인덱스 표시
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
g_idx =13
print(generated_images[g_idx])
# 1. NumPy 배열로 이미지 생성 (예: 흑백 이미지)
# 흑백 이미지 (값 범위: 0~255)

# 2. 이미지 크기를 정확히 200x200 픽셀로 저장
fig = plt.figure(figsize=(2, 2), dpi=100)  # figsize=(2, 2)와 dpi=100을 설정
ax = fig.add_subplot(111)
ax.imshow(generated_images[13], cmap='hot')
ax.axis('off')  # 축 제거
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)  # 여백 제거
plt.savefig("./compare_result/CNN.png", dpi=100, bbox_inches='tight', pad_inches=0)
plt.close()

In [ ]:
###################################
#Make Model input from local file.#
###################################

def meteo_extend(input, size) :
    # Step 2: 배열을 (1, 1, 25) 형상으로 변환
    #arr_expanded = np.expand_dims(np.expand_dims(input, axis=0), axis=0)  # (1, 1, 25)

    # Step 3: 이 배열을 (512, 512, 25)로 확장
    expanded_arr = np.broadcast_to(input, (image_size, image_size, 1))
    print(expanded_arr)
    # Step 4: 이 텐서를 500번 반복하여 (n, 512, 512, 1) 텐서 생성
    return np.stack([expanded_arr] * size)
    
#1~900 real count 874, 901~1701 real count 785

ws1 = np.stack([4.6/14.1*4 -2]*1)
wd1 = np.stack([313/360*4 -2]*1)
temp1 = np.stack([74/78*0.5 +0.5]*1)
hume1 = np.stack([29/61*4-2]*1)


real_wind_vector = np.array([[[ws], [wd]] for ws, wd in zip(ws1, wd1)])
real_tnh = np.array([[[temp, hume]] for temp, hume in zip(temp1, hume1)])



real_firestate = fire_state[0:1]
real_fuel = fuel[0:1]
real_dem = dem[0:1]






# noise 생성

generated_image = model.predict([real_firestate, real_fuel, real_dem, real_wind_vector, real_tnh])
generated_image = generated_image[0]
generated_image = 0.25 * generated_image + 0.5
generated_image = np.squeeze(generated_image, axis=2)

plt.imshow(generated_image, cmap='hot')
plt.show()

In [ ]:
def calculate_mse(img1, img2):
    return np.mean((img1 - img2) ** 2)

def evaluate_model(model, dataset):
    max_mse = 0
    min_ssim = float('inf')  # SSIM의 최소값을 찾기 위해 초기값을 무한대로 설정
    min_mse = float('inf')  # MSE의 최소값을 찾기 위해 초기값을 무한대로 설정

    max_mse_case = None
    min_mse_case = None
    min_ssim_case = None
    total_mse = 0
    total_ssim = 0
    count = 0
    mse_all = []
    for batch in dataset:
        firestate, fuel, dem, wind_vector, tnh = batch[0]
        sstdf = batch[-1]

        # 배치 크기에 맞는 fake_tensor 생성
        batch_size = firestate.shape[0]

        # Generator 예측 (perimeter 자리에 fake_tensor 사용)
        try:
            prediction = model.predict([firestate, fuel, dem, wind_vector, tnh])
        except Exception as e:
            print(f"Error during prediction: {e}")
            break

        for i in range(batch_size):
            real_image = sstdf[i]  # 실제 이미지 (실제 Y 값)
            pred_image = prediction[i]   # 예측된 이미지 (Generator의 출력)
            pred_image = np.squeeze(pred_image, axis=2)

            mse_value = calculate_mse(real_image, pred_image)
            mse_all.append(mse_value)
            total_mse += mse_value
            count += 1

            # 최대 MSE 확인
            if mse_value > max_mse:
                max_mse = mse_value
                max_mse_case = (real_image, pred_image)
            
            # 최소 MSE 확인
            if mse_value < min_mse:
                min_mse = mse_value
                min_mse_case = (real_image, pred_image)


    avg_mse = total_mse / count


    return max_mse, avg_mse, max_mse_case, min_mse ,min_mse_case, mse_all 


def plot_comparison(real_image, pred_image, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(real_image, cmap='hot')
    axes[0].set_title('Real Image')
    axes[1].imshow(pred_image, cmap='hot')
    axes[1].set_title('Predicted Image')
    plt.suptitle(title)
    plt.show()


In [ ]:

# Train 데이터셋에 대한 평가
train_max_mse, train_avg_mse, train_max_mse_case, train_min_mse, train_min_mse_case, train_mse_all = evaluate_model(model, train_dataset)

# Test 데이터셋에 대한 평가
test_max_mse, test_avg_mse, test_max_mse_case, test_min_mse, test_min_mse_case, test_mse_all = evaluate_model(model, test_dataset)




In [ ]:
# 결과 출력
print(f"Train Dataset: Avg MSE = {train_avg_mse}, max:{train_max_mse}")
print(f"Test Dataset: Avg MSE = {test_avg_mse}, max:{test_max_mse}")

# 최대 MSE 케이스 시각화
real_image, pred_image = train_max_mse_case
plot_comparison(real_image, pred_image, "Train Dataset - Max MSE Case")

real_image, pred_image = test_max_mse_case
plot_comparison(real_image, pred_image, "Test Dataset - Max MSE Case")

# 최대 SSIM 케이스 시각화
real_image, pred_image = train_min_mse_case
plot_comparison(real_image, pred_image, "Train Dataset - Min MSE Case")

real_image, pred_image = test_min_mse_case
plot_comparison(real_image, pred_image, "Test Dataset - Min MSE Case")

In [ ]:
# 특정 값 설정
threshold = 0.00003

# threshold보다 큰 값의 비율을 계산
train_above = (np.sum(np.array(train_mse_all) > threshold) / len(train_mse_all)) * 100
test_above = (np.sum(np.array(test_mse_all) > threshold) / len(test_mse_all)) * 100
print(train_above)
print(test_above)

# 데이터를 정렬
sorted_data1 = np.sort(train_mse_all)
sorted_data2 = np.sort(test_mse_all)

# 데이터의 누적 비율 계산
cumulative_percent1 = np.arange(1, len(sorted_data1) + 1) / len(sorted_data1) * 100
cumulative_percent2 = np.arange(1, len(sorted_data2) + 1) / len(sorted_data2) * 100

# 그래프 그리기
plt.figure(figsize=(8, 6))
plt.plot(sorted_data1, cumulative_percent1, label="MSE for train_data")
plt.plot(sorted_data2, cumulative_percent2, label="MSE for test_data")
plt.axhline(y=100, color='r', linestyle='--', label="100% Line")
plt.title('MSE between real and predicted(CDF)')
plt.xlabel('Value')
plt.ylabel('Percentage(%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.save('cnn_copy_2.h5')